In [ ]:
import sys
import os
from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "self-instruct").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Cannot find the project root")


project_root = find_project_root()
print(f"Project root: {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from huggingface_hub import ModelCard, list_models
from data.utils.utility import clean_markdown
from time import sleep
import datetime
import json
import re


tasks_file = project_root / "self-instruct" / "data" / "tasks"
with open(tasks_file, "r") as f:
    t = json.load(f)

tasks = list(t.keys())
print(tasks)

In [ ]:
limit = 200
# limit_per_task = 100
min_year = 2024
collected_models = []
seen_ids = set()

In [ ]:
def parse_created_at(model):
    created_at = model.created_at or model.cardData.get("createdAt")
    if isinstance(created_at, str):
        return datetime.fromisoformat(created_at.replace("Z", "+00:00"))
    return created_at


def is_recent(model, min_year=2024, min_month=8):
    try:
        created_at = parse_created_at(model)
        if not created_at:
            return False
        return created_at.year > min_year or (
            created_at.year == min_year and created_at.month > min_month
        )
    except Exception:
        return False


def load_modelcard(model_id):
    try:
        card = ModelCard.load(model_id)
        return card.text
    except Exception:
        return None


count = 0
for task in tasks:
    task_count = 0
    print(f"\nSearch: task='{task}'")

    try:
        models = list_models(filter=task, sort="downloads", direction=-1, limit=limit)

        for model in models:
            if model.id in seen_ids:
                continue
            # TODO: change to is_old to get older models
            if not is_recent(model, min_year):
                continue

            created_at = parse_created_at(model)
            if not created_at:
                continue

            modelcard = load_modelcard(model.id)
            if not modelcard:
                continue

            modelcard = clean_markdown(modelcard)
            if len(modelcard.split()) <= 200:
                continue

            collected_models.append(
                {
                    "model_id": model.id,
                    "created_at": created_at.isoformat(),
                    "downloads": model.downloads,
                    "likes": model.likes,
                    "author": model.author,
                    "tags": model.tags,
                    "modelcard": modelcard,
                    "domain": task,
                }
            )

            seen_ids.add(model.id)
            task_count += 1
    except Exception as e:
        print(f"Error for task={task}: {e}")

    sleep(4)
    print(f"Found {len(collected_models) - count} models for tag '{task}'")
    count = len(collected_models)


In [ ]:
import re
import unicodedata


def normalize_text(text: str) -> str:
    """
    Normalize text for robust substring matching.
    """
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Case folding (stronger than lower())
    text = text.casefold()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def is_autogenerated_modelcard(modelcard: str) -> bool:
    """
    Returns True if the normalized modelcard contains
    'This model card has been automatically generated.'.
    """
    if not modelcard:
        return False

    target = normalize_text("This model card has been automatically generated.")
    content = normalize_text(modelcard)

    return target in content


print(f"Before filtering, {len(collected_models)} entries found.")
# Filter out autogenerated model cards
filtered_models = [
    entry
    for entry in collected_models
    if not is_autogenerated_modelcard(entry.get("modelcard", ""))
]
print(f"After filtering, {len(filtered_models)} entries remain.")

In [ ]:
import pandas as pd

df = pd.DataFrame(collected_models)
df.groupby("domain").count().sort_values(by="model_id", ascending=False)

In [ ]:
def remove_duplicates_by_modelcard(models, key="modelcard", priority_key="downloads"):
    """
    Remove duplicate models based on a key (e.g., 'modelcard').
    When duplicates are found, keep the one with the highest value for priority_key.

    Args:
        models: List of model dictionaries
        key: Field to check for duplicates (default: 'modelcard')
        priority_key: Field to use for selecting which duplicate to keep (default: 'downloads')

    Returns:
        List of unique models
    """
    unique_items = {}

    for item in models:
        key_value = item.get(key)

        if key_value is None:
            # Skip items without the key
            continue

        if key_value not in unique_items:
            # First time seeing this key value
            unique_items[key_value] = item
        else:
            # Duplicate found - keep the one with higher priority_key value
            current_priority = item.get(priority_key, 0)
            existing_priority = unique_items[key_value].get(priority_key, 0)

            if current_priority > existing_priority:
                unique_items[key_value] = item

    return list(unique_items.values())


cleaned_models = remove_duplicates_by_modelcard(
    filtered_models, key="modelcard", priority_key="downloads"
)
print(f"After removing duplicates, {len(cleaned_models)} entries remain.")

In [ ]:
df = pd.DataFrame(cleaned_models)
# drop domain == any-to-any
df = df[df["domain"] != "any-to-any"]
# keep 25 most downloaded models per domain
df = df.sort_values(by="downloads", ascending=False).groupby("domain").head(25)
cleaned_models = df.to_dict(orient="records")
cleaned_models

In [ ]:
# save collected_models to jsonl
output_dir = project_root / "self-instruct" / "data"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "model_cards_step_1.jsonl"
with open(output_file, "w") as f:
    for item in cleaned_models:
        f.write(json.dumps(item) + "\n")
# read collected_models from jsonl